# Organizational Graph Pipeline - Test Notebook

This notebook tests the integrated pipeline step by step:
1. **Load data** from Excel files (organizational units + activities)
2. **Build org hierarchy graph** with activity nodes
3. **Vectorize** activity descriptions in vector store
4. **Upload** graph to FalkorDB
5. **Save** local JSON backup
6. **Verify** FalkorDB contents

In [20]:
# =============================================================================
# CONFIGURATION
# =============================================================================

import os
import sys
from dotenv import load_dotenv

sys.path.insert(0, '.')

load_dotenv()

ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")

# ═══════════════════════════════════════════════════════════════════════════
# DATA SOURCE
# ═══════════════════════════════════════════════════════════════════════════

DATA_SOURCE_PATH = os.getenv("DATA_SOURCE_PATH", "./data/samples/CASP_Activities_EN.xlsx")

# ═══════════════════════════════════════════════════════════════════════════
# GRAPH STRUCTURE OPTIONS
# ═══════════════════════════════════════════════════════════════════════════

ACTIVITIES_AS_NODES = True
ACTIVITY_ID_PREFIX = "ACT"
MASTER_ROOT_ID = "ORG_MASTER"
MASTER_ROOT_NAME = "European Parliament Organizations"

# ═══════════════════════════════════════════════════════════════════════════
# VECTOR STORE OPTIONS
# ═══════════════════════════════════════════════════════════════════════════

VECTOR_STORE_TYPE = os.getenv("VECTOR_STORE_TYPE", "mock")
VECTOR_PERSIST_DIR = os.getenv("VECTOR_PERSIST_DIR", "./chroma_db")
VECTOR_COLLECTION_NAME = os.getenv("VECTOR_COLLECTION_NAME", "org_documents")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "all-MiniLM-L6-v2")

# ═══════════════════════════════════════════════════════════════════════════
# GRAPH STORE OPTIONS — FALKORDB
# ═══════════════════════════════════════════════════════════════════════════

GRAPH_STORE_TYPE = "falkordb"
USE_FALKORDB = True

FALKORDB_URL = os.getenv("FALKORDB_URL", "")
FALKORDB_GRAPH_NAME = os.getenv("FALKORDB_GRAPH_NAME", "org_hierarchy")
GRAPH_LOCAL_PATH = os.getenv("GRAPH_LOCAL_PATH", "./graph_data/organization_graph.json")

if USE_FALKORDB and not FALKORDB_URL:
    raise EnvironmentError("FALKORDB_URL must be set in .env when USE_FALKORDB is enabled.")

# ═══════════════════════════════════════════════════════════════════════════
# PIPELINE FLAGS
# ═══════════════════════════════════════════════════════════════════════════

CLEAR_EXISTING = True
CREATE_VECTOR_INDEX = True
SAVE_LOCAL = True
VERBOSE = True
SHOW_ACTIVITIES_IN_TREE = False

# ═══════════════════════════════════════════════════════════════════════════
# STARTUP SUMMARY
# ═══════════════════════════════════════════════════════════════════════════

print("Configuration:")
print(f"  Data source:        {DATA_SOURCE_PATH}")
print(f"  Activities as nodes:{ACTIVITIES_AS_NODES}")
print(f"  Vector store:       {VECTOR_STORE_TYPE}")
print(f"  Embedding model:    {EMBEDDING_MODEL}")
print(f"  Graph store:        {GRAPH_STORE_TYPE}")
print(f"  Use FalkorDB:       {USE_FALKORDB}")
masked = FALKORDB_URL.split("@")[-1] if "@" in FALKORDB_URL else "(set)"
print(f"  FalkorDB host:      {masked}")

Configuration:
  Data source:        ./data/samples/CASP_Activities_EN.xlsx
  Activities as nodes:True
  Vector store:       mock
  Embedding model:    all-MiniLM-L6-v2
  Graph store:        falkordb
  Use FalkorDB:       True
  FalkorDB host:      r-6jissuruar.instance-zyadijt4h.hc-v8noonp0c.europe-west1.gcp.f2e0a955bb84.cloud:55831


---
## Step 1: Create Pipeline

In [21]:
# =============================================================================
# CREATE PIPELINE
# =============================================================================

from pipeline import create_pipeline

pipeline = create_pipeline(
    data_source_path=DATA_SOURCE_PATH,
    activities_as_nodes=ACTIVITIES_AS_NODES,
    vector_store_type=VECTOR_STORE_TYPE,
    graph_store_type=GRAPH_STORE_TYPE,
    falkordb_url=FALKORDB_URL,
    verbose=VERBOSE
)

print(f"\n✓ Pipeline created with {GRAPH_STORE_TYPE} graph store")


✓ Pipeline created with falkordb graph store


In [22]:
# =============================================================================
# INITIALIZE PIPELINE STORES
# =============================================================================

pipeline._init_stores()

print(f"✓ Vector store: {type(pipeline.vector_store).__name__}")
print(f"✓ Graph store:  {type(pipeline.graph_store).__name__}")
print(f"✓ Data source:  {type(pipeline.data_source).__name__}")

✓ VectorStore: Mock (in-memory)
✓ GraphStore: FalkorDB (org_hierarchy)
✓ Vector store: MockVectorStore
✓ Graph store:  FalkorDBGraphStore
✓ Data source:  ExcelDataSource


In [23]:
# =============================================================================
# DEBUG: FALKORDB CONNECTION
# =============================================================================

print("1. Checking graph store type...")
print(f"   Type: {type(pipeline.graph_store).__name__}")

print("\n2. Checking connection status...")
print(f"   _connected: {getattr(pipeline.graph_store, '_connected', 'N/A')}")
print(f"   _client:    {pipeline.graph_store._client}")
print(f"   _graph:     {pipeline.graph_store._graph}")

print("\n3. Checking config...")
print(f"   falkordb_url set: {bool(pipeline.graph_store.config.falkordb_url)}")
masked = pipeline.graph_store.config.falkordb_url.split("@")[-1] if pipeline.graph_store.config.falkordb_url and "@" in pipeline.graph_store.config.falkordb_url else "(empty)"
print(f"   falkordb_url host: {masked}")
print(f"   graph_name: {pipeline.graph_store.config.graph_name}")

print("\n4. Attempting connect()...")
result = pipeline.graph_store.connect()
print(f"   connect() returned: {result}")
print(f"   _connected after:   {pipeline.graph_store._connected}")

if pipeline.graph_store._connected:
    print("\n5. Testing write + read...")
    pipeline.graph_store._graph.query("CREATE (t:Test {node_id: 'test_123', name: 'connection_test'})")
    check = pipeline.graph_store._graph.query("MATCH (t:Test {node_id: 'test_123'}) RETURN t.name as name")
    print(f"   Write/read test: {check.result_set}")
    pipeline.graph_store._graph.query("MATCH (t:Test {node_id: 'test_123'}) DELETE t")
    print("   Cleanup: OK")
else:
    print("\n⚠️ Connection failed — check FALKORDB_URL in .env")

1. Checking graph store type...
   Type: FalkorDBGraphStore

2. Checking connection status...
   _connected: True
   _client:    <falkordb.falkordb.FalkorDB object at 0x000002813F399850>
   _graph:     <falkordb.graph.Graph object at 0x000002813F3CF250>

3. Checking config...
   falkordb_url set: True
   falkordb_url host: r-6jissuruar.instance-zyadijt4h.hc-v8noonp0c.europe-west1.gcp.f2e0a955bb84.cloud:55831
   graph_name: org_hierarchy

4. Attempting connect()...
✓ GraphStore: FalkorDB (org_hierarchy)
   connect() returned: True
   _connected after:   True

5. Testing write + read...
   Write/read test: [['connection_test']]
   Cleanup: OK


---
## Step 2: Define Org Hierarchy Parser & Step Functions

In [24]:
# =============================================================================
# ORG HIERARCHY PARSER — handles all EP code patterns
# =============================================================================
import re

def parse_org_hierarchy(entity_codes):
    """
    Parse EP entity codes into parent-child relationships.

    Patterns:
      '22'       → top-level DG (root)
      '22-10'    → direct unit under DG (dash = attached unit)
      '22A'      → directorate under DG
      '22A10'    → unit under directorate 22A
      '22B0040'  → sub-unit under 22B00 (implicit parent synthesized)
    """
    codes = sorted(set(entity_codes))
    hierarchy = {}

    for code in codes:
        # Root DG — two digits only
        if re.match(r'^\d{2}$', code):
            hierarchy[code] = None

        # Dash-attached unit (e.g., '22-10') → parent is DG
        elif re.match(r'^(\d{2})-\d+$', code):
            dg = re.match(r'^(\d{2})', code).group(1)
            hierarchy[code] = dg

        # Directorate (e.g., '22A') → parent is DG
        elif re.match(r'^\d{2}[A-Z]$', code):
            hierarchy[code] = code[:2]

        # Unit under directorate (e.g., '22A10') → parent is directorate
        elif re.match(r'^\d{2}[A-Z]\d{2}$', code):
            hierarchy[code] = code[:3]

        # Sub-unit with 4+ digits (e.g., '22B0040') → synthesize parent
        elif re.match(r'^\d{2}[A-Z]\d{4,}$', code):
            directorate = code[:3]
            parent_code = code[:5]
            if parent_code not in codes and parent_code not in hierarchy:
                hierarchy[parent_code] = directorate
            hierarchy[code] = parent_code

        else:
            dg = re.match(r'^(\d{2})', code).group(1)
            hierarchy[code] = dg
            print(f"  ⚠️ Unrecognized code pattern '{code}', attached to root {dg}")

    return hierarchy

# --- Quick test ---
codes = ['22', '22-10', '22A', '22A10', '22A20', '22A30', '22A40',
         '22B', '22B0040', '22B10', '22B20', '22B30']

hierarchy = parse_org_hierarchy(codes)
print("Parent-child mapping:")
for child, parent in sorted(hierarchy.items()):
    print(f"  {child:10s} → {parent}")

Parent-child mapping:
  22         → None
  22-10      → 22
  22A        → 22
  22A10      → 22A
  22A20      → 22A
  22A30      → 22A
  22A40      → 22A
  22B        → 22
  22B00      → 22B
  22B0040    → 22B00
  22B10      → 22B
  22B20      → 22B
  22B30      → 22B


In [25]:
# =============================================================================
# PIPELINE STEP FUNCTIONS
# =============================================================================
import pandas as pd
import networkx as nx


def step_load_data(filepath):
    """Step 1: Load data — each row is one activity."""
    print("=" * 60)
    print("STEP 1: LOAD DATA")
    print("=" * 60)

    df = pd.read_excel(filepath)

    # Row number in Excel (1-based)
    df['Excel_Row'] = range(1, len(df) + 1)

    # Synthetic activity ID: ORG_UNIT_ID.ROW_NUMBER
    df['Activity_ID'] = df['Entity Code'] + '.' + df['Excel_Row'].astype(str)

    print(f"  Total activities (rows): {len(df)}")
    print(f"  Org units: {df['Entity Code'].nunique()}")
    print(f"  Sample activity IDs:")
    for _, row in df.head(5).iterrows():
        print(f"    {row['Activity_ID']:15s} ({row['Entity Code']})")

    display(df[['Activity_ID', 'Entity Code',
                 'Key Activities (Missions principales)', '%']].head(5))

    return df


def step_build_org_graph(df):
    """Step 2: Build org hierarchy + one node per activity."""
    print("=" * 60)
    print("STEP 2: BUILD ORG HIERARCHY + ACTIVITY NODES")
    print("=" * 60)

    G = nx.DiGraph()

    # --- Org unit nodes + hierarchy edges ---
    entity_codes = df['Entity Code'].unique()
    code_to_name = dict(zip(df['Entity Code'], df['Entity Name']))
    hierarchy = parse_org_hierarchy(entity_codes)

    for code, parent in hierarchy.items():
        G.add_node(code,
                    type='org_unit',
                    node_type='organization',
                    name=code_to_name.get(code, f'(synthesized) {code}'),
                    level=sum(1 for _ in nx.ancestors(G, code)) if code in G else 0)
        if parent is not None:
            G.add_edge(parent, code, edge_type='HAS_CHILD')

    org_count = G.number_of_nodes()

    # --- Activity nodes: one per Excel row ---
    for _, row in df.iterrows():
        activity_id = row['Activity_ID']
        entity_code = row['Entity Code']

        G.add_node(activity_id,
                    type='activity',
                    node_type='activity',
                    description=row['Key Activities (Missions principales)'],
                    weight_pct=row['%'],
                    parent_org=entity_code,
                    excel_row=row['Excel_Row'])

        G.add_edge(entity_code, activity_id, edge_type='HAS_ACTIVITY')

    activity_count = G.number_of_nodes() - org_count

    print(f"  Org unit nodes:    {org_count}")
    print(f"  Activity nodes:    {activity_count}")
    print(f"  Total nodes:       {G.number_of_nodes()}")
    print(f"  Total edges:       {G.number_of_edges()}")

    synthesized = [n for n, d in G.nodes(data=True)
                   if d.get('type') == 'org_unit' and '(synthesized)' in d.get('name', '')]
    if synthesized:
        print(f"  Synthesized parents: {synthesized}")

    return G


def step_vectorize(pipeline, df):
    """Step 3: Vectorize activity descriptions in vector store."""
    print("=" * 60)
    print("STEP 3: VECTORIZE ACTIVITIES")
    print("=" * 60)

    documents = []
    ids = []
    for _, row in df.iterrows():
        documents.append({
            'content': row['Key Activities (Missions principales)'],
            'metadata': {
                'entity_code': row['Entity Code'],
                'entity_name': row['Entity Name'],
                'weight_pct': row['%'],
                'excel_row': row['Excel_Row']
            }
        })
        ids.append(row['Activity_ID'])

    pipeline.vector_store.add_documents(documents, ids=ids)
    count = pipeline.vector_store.count()
    print(f"  Documents vectorized: {count}")

    sample = documents[0]['content'][:80]
    results = pipeline.vector_store.search(sample, k=3)
    print(f"  Sanity query: '{sample}...'")
    print(f"  Top matches returned: {len(results)}")

    return documents


def step_upload_to_graph_store(pipeline, graph):
    """Step 4: Upload graph to FalkorDB (or NetworkX)."""
    print("=" * 60)
    print("STEP 4: UPLOAD TO GRAPH STORE")
    print("=" * 60)

    # Clear existing data
    if CLEAR_EXISTING:
        print("  Clearing existing graph data...")
        pipeline.graph_store.clear()

    # Add master root node
    pipeline.graph_store.add_node(
        MASTER_ROOT_ID,
        name=MASTER_ROOT_NAME,
        level=-1,
        node_type='master_root',
        is_master_root=True
    )

    # Upload the full graph
    count = pipeline.graph_store.upload_graph(graph)

    # Connect org roots to master root
    roots = [n for n, d in graph.nodes(data=True)
             if d.get('type') == 'org_unit' and graph.in_degree(n) == 0]
    for root_id in roots:
        pipeline.graph_store.add_edge(MASTER_ROOT_ID, root_id, edge_type='HAS_ORGANIZATION')
        print(f"  Connected {MASTER_ROOT_ID} → {root_id}")

    # Create vector index if FalkorDB
    if CREATE_VECTOR_INDEX and hasattr(pipeline.graph_store, 'create_vector_index'):
        pipeline.graph_store.create_vector_index(
            index_name="org_embedding_idx",
            node_label="OrganizationalUnit",
            property_name="embedding",
            dimension=384
        )

    stats = pipeline.graph_store.get_stats()
    print(f"\n  Upload complete:")
    print(f"  Nodes in store: {stats.get('nodes', count)}")
    print(f"  Edges in store: {stats.get('edges', 0)}")
    print(f"  Backend:        {stats.get('backend', 'unknown')}")

    return stats


def step_save_locally(pipeline, graph):
    """Step 5: Save graph to local JSON as backup."""
    print("=" * 60)
    print("STEP 5: SAVE LOCAL BACKUP")
    print("=" * 60)

    if hasattr(pipeline.graph_store, 'save'):
        # NetworkXGraphStore has save()
        pipeline.graph_store.graph = graph
        output_path = pipeline.graph_store.save()
    else:
        # FalkorDB or other — save NetworkX graph directly as JSON
        import json
        from pathlib import Path

        graph_data = {
            "metadata": {
                "nodes": graph.number_of_nodes(),
                "edges": graph.number_of_edges(),
                "master_root_id": MASTER_ROOT_ID,
                "backend": GRAPH_STORE_TYPE
            },
            "nodes": [
                {"id": n, **{k: v for k, v in d.items()}}
                for n, d in graph.nodes(data=True)
            ],
            "edges": [
                {"source": u, "target": v, **d}
                for u, v, d in graph.edges(data=True)
            ]
        }

        output_path = GRAPH_LOCAL_PATH
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(graph_data, f, indent=2, ensure_ascii=False, default=str)

        print(f"  Saved to: {output_path}")

    return output_path


print("✓ All step functions defined")

✓ All step functions defined


---
## Step 3: Run Pipeline Steps

In [26]:
# =============================================================================
# STEP 1: LOAD DATA
# =============================================================================

data = step_load_data(DATA_SOURCE_PATH)

STEP 1: LOAD DATA
  Total activities (rows): 78
  Org units: 12
  Sample activity IDs:
    22.1            (22)
    22.2            (22)
    22.3            (22)
    22.4            (22)
    22.5            (22)


,Activity_ID,Entity Code,Key Activities (Missions principales),%
0,22.1,22,Define strategy and ensure the development of ...,25
1,22.2,22,Ensure the proper functioning and administrati...,25
2,22.3,22,Represent the general management within the bo...,15
3,22.4,22,Manage human resources and the general managem...,15
4,22.5,22,Represent general management in interinstituti...,10


In [27]:
# =============================================================================
# STEP 2: BUILD ORG HIERARCHY GRAPH
# =============================================================================

graph = step_build_org_graph(data)

STEP 2: BUILD ORG HIERARCHY + ACTIVITY NODES
  Org unit nodes:    13
  Activity nodes:    78
  Total nodes:       91
  Total edges:       90
  Synthesized parents: ['22B00']


In [28]:
# =============================================================================
# INSPECT: ORG UNITS AND THEIR ACTIVITIES
# =============================================================================

print("--- Org Units and their Activities ---")
for node, attrs in sorted(graph.nodes(data=True)):
    if attrs['type'] == 'org_unit':
        children = list(graph.successors(node))
        activities = [c for c in children if graph.nodes[c]['type'] == 'activity']
        sub_orgs = [c for c in children if graph.nodes[c]['type'] == 'org_unit']
        print(f"  {node:10s} | {len(sub_orgs)} sub-orgs | {len(activities)} activities "
              f"| IDs: {[a for a in activities[:3]]}{'...' if len(activities) > 3 else ''}")

--- Org Units and their Activities ---
  22         | 3 sub-orgs | 7 activities | IDs: ['22.1', '22.2', '22.3']...
  22-10      | 0 sub-orgs | 8 activities | IDs: ['22-10.8', '22-10.9', '22-10.10']...
  22A        | 4 sub-orgs | 6 activities | IDs: ['22A.16', '22A.17', '22A.18']...
  22A10      | 0 sub-orgs | 6 activities | IDs: ['22A10.22', '22A10.23', '22A10.24']...
  22A20      | 0 sub-orgs | 6 activities | IDs: ['22A20.28', '22A20.29', '22A20.30']...
  22A30      | 0 sub-orgs | 6 activities | IDs: ['22A30.34', '22A30.35', '22A30.36']...
  22A40      | 0 sub-orgs | 6 activities | IDs: ['22A40.40', '22A40.41', '22A40.42']...
  22B        | 4 sub-orgs | 6 activities | IDs: ['22B.46', '22B.47', '22B.48']...
  22B00      | 1 sub-orgs | 0 activities | IDs: []
  22B0040    | 0 sub-orgs | 5 activities | IDs: ['22B0040.74', '22B0040.75', '22B0040.76']...
  22B10      | 0 sub-orgs | 6 activities | IDs: ['22B10.52', '22B10.53', '22B10.54']...
  22B20      | 0 sub-orgs | 8 activities | IDs: ['

In [29]:
# =============================================================================
# STEP 3: VECTORIZE ACTIVITIES
# =============================================================================

documents = step_vectorize(pipeline, data)

STEP 3: VECTORIZE ACTIVITIES
  Documents vectorized: 78
  Sanity query: 'Define strategy and ensure the development of general management. Contribute to ...'
  Top matches returned: 3


In [30]:
# =============================================================================
# STEP 4: UPLOAD TO FALKORDB
# =============================================================================

store_results = step_upload_to_graph_store(pipeline, graph)

STEP 4: UPLOAD TO GRAPH STORE
  Clearing existing graph data...
  Uploaded 91 nodes to FalkorDB
  Connected ORG_MASTER → 22
⚠ Vector index creation: errMsg: Invalid input 'o': expected CREATE INDEX FOR line: 2, column: 33, offset: 33 errCtx:             CREATE VECTOR INDEX org_embedding_idx errCtxOffset: 32

  Upload complete:
  Nodes in store: 92
  Edges in store: 91
  Backend:        falkordb


In [31]:
# =============================================================================
# STEP 5: SAVE LOCAL BACKUP
# =============================================================================

output_path = step_save_locally(pipeline, graph)

STEP 5: SAVE LOCAL BACKUP
  Saved to: ./graph_data/organization_graph.json


---
## Step 4: Verify FalkorDB Contents

In [32]:
# =============================================================================
# VERIFY FALKORDB CONTENTS
# =============================================================================

print("=" * 60)
print("FALKORDB VERIFICATION")
print("=" * 60)

try:
    org_count = pipeline.graph_store.query(
        "MATCH (n:OrganizationalUnit) RETURN count(n) as count"
    )
    dg_count = pipeline.graph_store.query(
        "MATCH (n:DG) RETURN count(n) as count"
    )
    act_count = pipeline.graph_store.query(
        "MATCH (n:Activity) RETURN count(n) as count"
    )
    edge_count = pipeline.graph_store.query(
        "MATCH ()-[r]->() RETURN type(r) as rel_type, count(r) as count"
    )

    print(f"  DG nodes:              {dg_count[0].get('count', 0) if dg_count else 0}")
    print(f"  OrganizationalUnit:    {org_count[0].get('count', 0) if org_count else 0}")
    print(f"  Activity nodes:        {act_count[0].get('count', 0) if act_count else 0}")
    print(f"\n  Edge types:")
    for row in edge_count:
        print(f"    {row.get('rel_type', '?'):20s}  {row.get('count', 0)}")

except Exception as e:
    print(f"  Query error: {e}")

FALKORDB VERIFICATION
Query error: unhashable type: 'list'
Query error: unhashable type: 'list'
Query error: unhashable type: 'list'
Query error: unhashable type: 'list'
  DG nodes:              0
  OrganizationalUnit:    0
  Activity nodes:        0

  Edge types:


---
## Step 5: Query Sample Data from FalkorDB

In [33]:
# =============================================================================
# QUERY SAMPLE NODES FROM FALKORDB
# =============================================================================

print("=" * 60)
print("SAMPLE NODES FROM FALKORDB")
print("=" * 60)

try:
    # Get DG root node
    dg_nodes = pipeline.graph_store.query(
        "MATCH (n:DG) RETURN n.node_id as id, n.name as name LIMIT 5"
    )
    print("\nDG nodes:")
    for row in dg_nodes:
        print(f"  [{row.get('id')}] {row.get('name')}")

    # Get org units
    org_nodes = pipeline.graph_store.query(
        "MATCH (n:OrganizationalUnit) RETURN n.node_id as id, n.name as name LIMIT 10"
    )
    print("\nOrganizational Units (first 10):")
    for row in org_nodes:
        print(f"  [{row.get('id')}] {row.get('name')}")

    # Get activities for a specific org unit
    sample_org = dg_nodes[0].get('id') if dg_nodes else '22'
    activities = pipeline.graph_store.query(
        """MATCH (org {node_id: $org_id})-[:HAS_ACTIVITY]->(act:Activity)
        RETURN act.node_id as id, act.description as description, act.weight_pct as weight
        LIMIT 5""",
        {"org_id": sample_org}
    )
    print(f"\nActivities for {sample_org} (first 5):")
    for row in activities:
        desc = str(row.get('description', ''))[:80]
        print(f"  [{row.get('id')}] ({row.get('weight')}%) {desc}...")

except Exception as e:
    print(f"  Query error: {e}")

SAMPLE NODES FROM FALKORDB
Query error: unhashable type: 'list'

DG nodes:
Query error: unhashable type: 'list'

Organizational Units (first 10):
Query error: unhashable type: 'list'

Activities for 22 (first 5):


---
## Step 6: Visualize Tree Structure

In [34]:
# =============================================================================
# DISPLAY ORGANIZATIONAL TREE (from local graph)
# =============================================================================

def print_tree(graph, master_root_id, show_activities=False):
    """Print the organizational tree."""

    def _print_node(node_id, prefix="", is_last=True):
        data = graph.nodes.get(node_id, {})
        name = data.get('name', data.get('description', node_id))
        if len(str(name)) > 50:
            name = str(name)[:50] + '...'
        node_type = data.get('type', 'unknown')

        connector = "└── " if is_last else "├── "

        if node_type == 'activity':
            weight = data.get('weight_pct', 0)
            print(f"{prefix}{connector}📋 ({weight}%) {name}")
        else:
            children = list(graph.successors(node_id))
            org_children = [c for c in children if graph.nodes[c].get('type') != 'activity']
            act_children = [c for c in children if graph.nodes[c].get('type') == 'activity']

            print(f"{prefix}{connector}[{node_id}] {name}")

            new_prefix = prefix + ("    " if is_last else "│   ")

            all_children = sorted(org_children)
            if show_activities:
                all_children += sorted(act_children)

            for i, child in enumerate(all_children):
                _print_node(child, new_prefix, i == len(all_children) - 1)

    print("\n" + "=" * 60)
    print("ORGANIZATIONAL TREE")
    print("=" * 60)
    print(f"[{master_root_id}] {MASTER_ROOT_NAME}")

    roots = [n for n, d in graph.nodes(data=True)
             if d.get('type') == 'org_unit' and graph.in_degree(n) == 0]

    for i, root in enumerate(sorted(roots)):
        _print_node(root, "    ", i == len(roots) - 1)

print_tree(graph, MASTER_ROOT_ID, show_activities=SHOW_ACTIVITIES_IN_TREE)


ORGANIZATIONAL TREE
[ORG_MASTER] European Parliament Organizations
    └── [22] Directorate-General for Cohesion, Agriculture and ...
        ├── [22-10] Resources Unit
        ├── [22A] Directorate for Regional Development, Agriculture ...
        │   ├── [22A10] Policy Department
        │   ├── [22A20] Secretariat of the Committee on Regional Developme...
        │   ├── [22A30] Secretariat of the Committee on Agriculture and Ru...
        │   └── [22A40] Secretariat of the Committee on Fisheries
        └── [22B] Directorate for Transport, Employment and Social A...
            ├── [22B00] (synthesized) 22B00
            │   └── [22B0040] Secretariat of the Special Committee on the Housin...
            ├── [22B10] Policy Department
            ├── [22B20] Secretariat of the Committee on Transport and Tour...
            └── [22B30] Secretariat of the Committee on Employment and Soc...


In [35]:
# =============================================================================
# TREE WITH ACTIVITIES
# =============================================================================

print_tree(graph, MASTER_ROOT_ID, show_activities=True)


ORGANIZATIONAL TREE
[ORG_MASTER] European Parliament Organizations
    └── [22] Directorate-General for Cohesion, Agriculture and ...
        ├── [22-10] Resources Unit
        │   ├── 📋 (15%) Develop the strategic planning and reporting cycle...
        │   ├── 📋 (10%) Organize internal training in cooperation with DG ...
        │   ├── 📋 (10%) Logistics: Ensure the logistical support necessary...
        │   ├── 📋 (10%) Coordinate, establish and draft the general manage...
        │   ├── 📋 (8%) Management of the budget envelope for missions (3L...
        │   ├── 📋 (7%) Coordinate internal communication (writing and pub...
        │   ├── 📋 (20%) Manage the recruitment of staff for the DG, the or...
        │   └── 📋 (20%) Manage the centralized financial system; ensure in...
        ├── [22A] Directorate for Regional Development, Agriculture ...
        │   ├── [22A10] Policy Department
        │   │   ├── 📋 (35%) Provide assistance to parliamentary bodies and the...
        │   

---
## Step 7: Test Semantic Search

In [36]:
# =============================================================================
# SEMANTIC SEARCH TEST
# =============================================================================

test_queries = [
    "budget management and financial oversight",
    "committee secretariat support",
    "fisheries and agriculture policy",
    "human resources recruitment",
    "legislative coordination",
]

print("=" * 60)
print("SEMANTIC SEARCH RESULTS")
print("=" * 60)

for query in test_queries:
    print(f"\nQuery: '{query}'")
    results = pipeline.vector_store.search(query, k=3)
    for i, result in enumerate(results):
        score = result.get('score', 0)
        content = result.get('content', '')[:80]
        entity = result.get('metadata', {}).get('entity_code', '?')
        print(f"  {i+1}. [{entity}] (score: {score:.3f}) {content}...")

SEMANTIC SEARCH RESULTS

Query: 'budget management and financial oversight'
  1. [22A] (score: 0.437) Ensure project management in the areas of expertise of the management units and ...
  2. [22B] (score: 0.437) Ensure project management in the areas of expertise of the management units and ...
  3. [22B0040] (score: 0.312) Prepare documents relating to the report (draft report, amendments document, etc...

Query: 'committee secretariat support'
  1. [22A20] (score: 0.337) Ensure follow-up of documents received as part of regional policy programming....
  2. [22B30] (score: 0.330) Ensure the secretariat of the commission (organization of meetings, coordinators...
  3. [22A30] (score: 0.304) Prepare working documents/draft reports (legislative and non-legislative procedu...

Query: 'fisheries and agriculture policy'
  1. [22A40] (score: 0.443) Prepare working documents/draft reports/opinions (legislative and non-legislativ...
  2. [22B30] (score: 0.434) Ensure contacts with the European

---
## Summary

In [37]:
# =============================================================================
# PIPELINE SUMMARY
# =============================================================================

print("\n" + "=" * 70)
print("PIPELINE SUMMARY")
print("=" * 70)

org_nodes = [n for n, d in graph.nodes(data=True) if d['type'] == 'org_unit']
act_nodes = [n for n, d in graph.nodes(data=True) if d['type'] == 'activity']

print(f"  Data source:       {DATA_SOURCE_PATH}")
print(f"  Org unit nodes:    {len(org_nodes)}")
print(f"  Activity nodes:    {len(act_nodes)}")
print(f"  Total graph nodes: {graph.number_of_nodes()}")
print(f"  Total graph edges: {graph.number_of_edges()}")
print(f"  Documents indexed: {pipeline.vector_store.count()}")
print(f"  Graph store:       {GRAPH_STORE_TYPE}")

if USE_FALKORDB:
    stats = pipeline.graph_store.get_stats()
    print(f"  FalkorDB nodes:    {stats.get('nodes', '?')}")
    print(f"  FalkorDB edges:    {stats.get('edges', '?')}")

print(f"  Local backup:      {GRAPH_LOCAL_PATH}")
print("\n✓ Pipeline complete")


PIPELINE SUMMARY
  Data source:       ./data/samples/CASP_Activities_EN.xlsx
  Org unit nodes:    13
  Activity nodes:    78
  Total graph nodes: 91
  Total graph edges: 90
  Documents indexed: 78
  Graph store:       falkordb
  FalkorDB nodes:    92
  FalkorDB edges:    91
  Local backup:      ./graph_data/organization_graph.json

✓ Pipeline complete


In [38]:
# =============================================================================
# CLEAR ALL NODES IN FALKORDB
# =============================================================================
"""
try:
    result = pipeline.graph_store._graph.query("MATCH (n) DETACH DELETE n")
    print(f"✓ All nodes deleted")
    
    # Verify
    check = pipeline.graph_store._graph.query("MATCH (n) RETURN count(n) as count")
    print(f"  Remaining nodes: {check.result_set[0][0]}")
except Exception as e:
    print(f"⚠️ Error: {e}")
"""

'\ntry:\n    result = pipeline.graph_store._graph.query("MATCH (n) DETACH DELETE n")\n    print(f"✓ All nodes deleted")\n\n    # Verify\n    check = pipeline.graph_store._graph.query("MATCH (n) RETURN count(n) as count")\n    print(f"  Remaining nodes: {check.result_set[0][0]}")\nexcept Exception as e:\n    print(f"⚠️ Error: {e}")\n'